# demo_rest_api — Loginom + GigaChat

Ноутбук проверяет пользователя через `CheckUserId`, получает TOP-10 через `GetUserHistory` и запрашивает рекомендацию у GigaChat. Секреты находятся только в `.env`.


In [ ]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv('.env')
LOGINOM_CHECK_URL = os.environ['LOGINOM_CHECK_URL']
LOGINOM_URL = os.environ['LOGINOM_URL']
GIGA_KEY = os.environ['GIGA_KEY']
USER_ID = 1  # замените на существующий ID
payload = {'Variables': {'user_id': USER_ID}}


## 1. Проверка существования пользователя


In [ ]:
check_response = requests.post(LOGINOM_CHECK_URL, json=payload, timeout=45)
check_response.raise_for_status()
check_data = check_response.json()
check_rows = check_data.get('DataSet', {}).get('Rows', [])
if not check_rows:
    raise RuntimeError('CheckUserId не вернул DataSet.Rows')
user_exists = str(check_rows[0].get('user_exists', 0)).strip().casefold() in {'1', 'true', 'yes', 'да'}
if not user_exists:
    raise ValueError(f'user_id={USER_ID} не найден')
print('Пользователь найден:', USER_ID)


## 2. Получение истории покупок


In [ ]:
history_response = requests.post(LOGINOM_URL, json=payload, timeout=45)
history_response.raise_for_status()
rows = history_response.json()['DataSet']['Rows']
if not rows:
    raise ValueError('Loginom вернул пустую историю покупок')
df = pd.DataFrame(rows)
display(df.head(10))


## 3. Формирование контекста


In [ ]:
lines = []
for i, row in enumerate(rows[:10], start=1):
    lines.append(f"{i}. {row['product_name']} — отдел: {row['department_rus']}, категория: {row['aisle']}, заказов: {row['order_count']}")
products_text = '\n'.join(lines)
print(products_text)


## 4. Запрос к GigaChat


In [ ]:
llm = GigaChat(credentials=GIGA_KEY, model='GigaChat-2', verify_ssl_certs=False, temperature=0.3, max_tokens=500)
system_prompt = '''Ты — персональный помощник покупателя в продуктовом онлайн-магазине.
Твоя задача — анализировать историю покупок клиента и давать ему понятные,
дружелюбные рекомендации прямо в его личном кабинете.
Обращайся к клиенту на "вы", тепло и по-человечески.
Пиши кратко — не более 5–6 предложений.'''
user_prompt = f'''Вот список товаров, которые клиент покупает чаще всего:\n\n{products_text}\n\nНапишите клиенту короткий персональный анализ его покупок. Начните с предпочтений и дайте 1–2 дружеских совета.'''
answer = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
print(answer.content)


## 5. Проверка

Повторите запрос с несуществующим `USER_ID`: `CheckUserId` должен вернуть `user_exists=0`, а ноутбук — остановиться до вызова `GetUserHistory`. Проверьте, что история содержит не более 10 товаров и относится к указанному пользователю.
